## Script to parse through the jsons of the politicians and store them in a local postgres db

In [1]:
from __future__ import annotations

import os
import re
import json
import hashlib
from dataclasses import dataclass
from datetime import date
from typing import Iterator, List, Optional, TypedDict, NewType, Literal, Any

import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from collections import Counter

from file_handling.file_read_writer import write_json, read_json
from params.paths import ROOT_DIR


LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(LOWER_HOUSE_DATA_DIR, 'historical')
LOWER_HOUSE_DATA_CURRENT = os.path.join(LOWER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_DATA_CURRENT = os.path.join(UPPER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(UPPER_HOUSE_DATA_DIR, 'historical')
os.makedirs(LOWER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)
os.makedirs(UPPER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)



In [2]:
PersonId = NewType("PersonId", int)

class RawElection(TypedDict):
    year: str          # e.g., "1952年"
    month: str         # e.g., "10月"
    day: str           # e.g., "1日"
    election_name: str
    district: str
    party: str
    result: str        # e.g., "当選"
    election_freq: str # e.g., "（1回目）"

class RawPersonFile(TypedDict):
    name_kanji: str
    name_kana: str
    election_data: List[RawElection]

@dataclass(frozen=True)
class Person:
    name_kanji: str
    name_kana: str
    election_signature: str

@dataclass(frozen=True)
class ElectionResult:
    person_id: PersonId
    election_date: date
    election_name: str
    district: str
    party: str
    result: str
    election_number: Optional[int]  # parsed from "（1回目）"
    raw_json: dict[str, Any]


In [3]:
DDL: list[str] = [
    """CREATE TABLE IF NOT EXISTS person (
        person_id BIGSERIAL PRIMARY KEY,
        name_kanji TEXT NOT NULL,
        name_kana  TEXT,
        election_signature TEXT,
        CONSTRAINT uq_person UNIQUE (name_kanji, name_kana, election_signature)
    );""",
    """CREATE TABLE IF NOT EXISTS election_result (
        id BIGSERIAL PRIMARY KEY,
        person_id BIGINT NOT NULL REFERENCES person(person_id),
        election_date DATE NOT NULL,
        election_name TEXT,
        district TEXT,
        party TEXT,
        result TEXT,
        election_number INT,
        raw_json JSONB,
        CONSTRAINT uq_election UNIQUE (person_id, election_date, election_name, district, party)
    );"""
]


In [4]:
DIGITS = re.compile(r"\d+")

def _parse_int(s: str) -> Optional[int]:
    m = DIGITS.search(s)
    return int(m.group(0)) if m else None

def parse_election_date(y: str, m: str, d: str) -> date:
    yy = _parse_int(y) or 1
    mm = _parse_int(m) or 1
    dd = _parse_int(d) or 1
    return date(yy, mm, dd)

def parse_election_number(freq: str) -> Optional[int]:
    # "（1回目）" -> 1
    return _parse_int(freq)

def generate_election_signature(raw: RawPersonFile) -> str:
    # Deterministic: hash sorted key/value pairs of the FIRST election entry
    # (Adjust to include more entries if you want a stronger signature)
    if not raw["election_data"]:
        base = f"{raw['name_kanji']}|{raw['name_kana']}|no_elections"
    else:
        first = raw["election_data"][0]
        parts = [f"{k}:{first[k]}" for k in sorted(first.keys())]
        base = f"{raw['name_kanji']}|{raw['name_kana']}|" + "|".join(parts)
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

In [5]:
# ---------- IO over your historical dirs ----------
def iterate_over_historical_data(house_historical_dir: str) -> Iterator[RawPersonFile]:
    for file in os.listdir(house_historical_dir):
        if not file.endswith(".json"):
            continue
        data = read_json(os.path.join(house_historical_dir, file))  # type: ignore[name-defined]
        # Lightweight runtime shape check; keep it cheap
        if not isinstance(data, dict):
            continue
        if not data.get("name_kana"):
            continue
        yield data  # type: ignore[typeddict-item]

# ---------- DB ops ----------
def run_ddl(cur: psycopg2.extensions.cursor) -> None:
    for stmt in DDL:
        cur.execute(stmt)

def insert_person(cur: psycopg2.extensions.cursor, p: Person) -> PersonId:
    cur.execute(
        """
        INSERT INTO person (name_kanji, name_kana, election_signature)
        VALUES (%s, %s, %s)
        ON CONFLICT (name_kanji, name_kana, election_signature) DO UPDATE
        SET name_kana = EXCLUDED.name_kana
        RETURNING person_id;
        """,
        (p.name_kanji, p.name_kana, p.election_signature),
    )
    pid = cur.fetchone()[0]
    return PersonId(pid)

def insert_elections_bulk(cur: psycopg2.extensions.cursor, rows: list[ElectionResult]) -> None:
    if not rows:
        return
    execute_values(
        cur,
        """
        INSERT INTO election_result (
            person_id, election_date, election_name, district, party, result, election_number, raw_json
        )
        VALUES %s
        ON CONFLICT DO NOTHING;
        """,
        [
            (
                int(r.person_id),
                r.election_date,
                r.election_name,
                r.district,
                r.party,
                r.result,
                r.election_number,
                json.dumps(r.raw_json)
            )
            for r in rows
        ],
    )

def upsert_person_and_elections(cur: psycopg2.extensions.cursor, raw: RawPersonFile) -> None:
    sig = generate_election_signature(raw)
    person = Person(
        name_kanji=raw["name_kanji"],
        name_kana=raw["name_kana"],
        election_signature=sig,
    )
    pid = insert_person(cur, person)

    results: list[ElectionResult] = []
    for e in raw["election_data"]:
        results.append(
            ElectionResult(
                person_id=pid,
                election_date=parse_election_date(e["year"], e["month"], e["day"]),
                election_name=e["election_name"],
                district=e["district"],
                party=e["party"],
                result=e["result"],
                election_number=parse_election_number(e["election_freq"]),
                raw_json=e,
            )
        )
    insert_elections_bulk(cur, results)

In [10]:
load_dotenv()

conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")

    with conn.cursor() as cur:
        run_ddl(cur)

        # Insert everything in one transaction
        # for hist_dir in [LOWER_HOUSE_DATA_CURRENT, UPPER_HOUSE_DATA_CURRENT]:
        hist_dir = os.path.join(ROOT_DIR, 'data', 'tmp', 'missed_reprs')
        for raw in iterate_over_historical_data(hist_dir):
            print(raw["name_kanji"])
            upsert_person_and_elections(cur, raw)

    conn.commit()
    print("Done.")

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()


Connected.
髙階恵美子
若林健太
宮本岳志
義家弘介
笠井亮
徳永久志
上野蛍
村上智信
衛藤征士郎
井原巧
前川清成
寺田学
高橋千鶴子
西川将人
大津力
金田勝年
Done.


## Checking whether all the duplicates in the db aare actual different people

In [6]:
from dotenv import load_dotenv
import psycopg2
from collections import Counter
import os
from typing import List, Dict, Tuple, Any

from api_requests.prompter import DeepResearchGemini

load_dotenv()

prompter = DeepResearchGemini()

duplication_check_sys_prompt = """
You are a helpful assistant that checks whether the politicians are the same person.
You will be given a list of politicians and their election data.
You will need to check whether they are the same person.
If they are the same person, you just respond "SAMEPERSON" We will pass the merging process to the next assistant.
If they are not the same person, you will respond "DIFFERENTPERSON". We will assume we do not have to merge the records.
"""

merging_sys_prompt = """
You are a helpful assistant that merges the election data of the politicians.
I will give you entries from my database that have the same name but different ids. These records were found to be duplicates by the previous assistant and need to be merged into a single, new record. Make sure you do research to verify the entries and create a new record that is accurate. You must supply your reply in the following format. Make sure you do not include any extra characters or comments or code blocks such as ```json.

{
	"name_kanji": "名前",
	"name_kana": "名前のふりがな",
	"years": [
	"当選年-当選月-当選日",
	"当選年-当選月-当選日",
	...
	],
	"election_data": [
		{
			"year": "当選年",
			"month": "当選月",
			"day": "当選日",
			"election_name": "当選回次",
			"district": "選挙区",
			"party": "政党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
	}

	返答フォーマットの例：
	{
		"name_kanji": "七条明",
		"name_kana": "しちじょうあきら",
		"years": [
			"1993-07-18",
			...
		],
		"election_data": [
			{
				"year": "1993年",
				"month": "7月",
				"day": "18日",
				"election_name": "第40回衆議院議員総選挙",
				"district": "徳島全県区",
				"party": "自由民主党",
				"result": "当選",
				"election_freq": "（1回目）"
			},
			...
		]
	}
"""

create_entry_sys_prompt = """
You are a helpful assistant that creates an entry for a politician in my database.
I will give you a politician's name and you will need to create an entry for them in my database. For the election history, make sure to only include federal elections.
Your response should be in the following format. Make sure you do not include any extra characters or comments or code blocks such as ```json.
{
	"name_kanji": "名前",
	"name_kana": "名前のふりがな",
	"years": [
	"当選年-当選月-当選日",
	"当選年-当選月-当選日",
	...
	],
	"election_data": [
		{
			"year": "当選年",
			"month": "当選月",
			"day": "当選日",
			"election_name": "当選回次",
			"district": "選挙区",
			"party": "政党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
}

返答フォーマットの例：
{
	"name_kanji": "七条明",
	"name_kana": "しちじょうあきら",
	"years": [
		"1993-07-18",
		...
	],
	"election_data": [
		{
			"year": "1993年",
			"month": "7月",
			"day": "18日",
			"election_name": "第40回衆議院議員総選挙",
			"district": "徳島全県区",
			"party": "自由民主党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
}
"""


def get_duplicate_politicians(cur: psycopg2.extensions.cursor):
    cur.execute("""
    SELECT p.*
    FROM public.person p
    JOIN (
        SELECT name_kanji
        FROM public.person
        GROUP BY name_kanji
        HAVING COUNT(*) > 1
    ) dup
    ON p.name_kanji = dup.name_kanji
    ORDER BY p.name_kanji;
    """)
    return cur.fetchall()

def create_name2ids_dict(duplicates:List):
    name2ids = {}
    for id, name, *_ in duplicates:
        if name not in name2ids:
            name2ids[name] = []
        name2ids[name].append(id)
    return name2ids

def compose_duplicate_check_prompt(name: str, id2elections:Dict[int, List[Tuple[str, Any]]]):
	prompt = f"""
	We have a DB with the following entries for politicians with the same name but different ids:	
	POLITICIAN NAME: {name}
	"""

	compose_election_record_str = lambda id, elections: f"""
	POLITICIAN ID: {id}
	ELECTION RECORD:
	{"\n".join([f"{'---'.join([str(e) for e in election[2:]])}" for election in elections])}
	"""
	for id, elections in id2elections.items():
		prompt += compose_election_record_str(id, elections)

	prompt += "\n\nCheck whether they are the same person by following the response format given in system prompt. Make sure you do some research on the internet if you need to."
	

	return prompt

def compose_merging_prompt(name:str, id2elections:Dict[int, List[Tuple[str, Any]]]):
	prompt = f"""
	POLITICIAN NAME: {name}
	"""

	compose_election_record_str = lambda id, elections: f"""
	POLITICIAN ID: {id}
	ELECTION RECORD:
	{"\n".join([f"{'---'.join([str(e) for e in election[2:]])}" for election in elections])}
	"""	
	for id, elections in id2elections.items():
		prompt += compose_election_record_str(id, elections)

	prompt += "\n\nMerge the election data of the politicians while verifying the data by doing some research on the internet."
	prompt += "\n\nMake sure you do not include any extra characters or comments or code blocks such as ```json."

	return prompt

def compose_create_entry_prompt(name:str, cur:psycopg2.extensions.cursor)->str:
	prompt = f"""
	Create an entry for the following politician in my database. Make sure to follow the format given in the system prompt.
	POLITICIAN NAME: {name}

	"""

	return prompt

def merge_and_update_db(name:str, id2elections:Dict[int, List[Tuple[str, Any]]], cur:psycopg2.extensions.cursor)->None:
	count = 0
	while True:
		try:
			prompt = compose_merging_prompt(name, id2elections)
			response, _,_ = prompter.prompt(prompt, merging_sys_prompt)
			response = response.replace("```json", "").replace("```", "")
			response = json.loads(response)
			# resume = input(f"""Do you want to resume the merging process? (y/n)\nNAME OF REPRESENTATIVE: {name}\nNEW ELECTION DATA: {response}""")
			# if resume == "n":
			# 	raise Exception("User decided to stop the merging process.")
			new_person = Person(
				name_kanji=response["name_kanji"],
				name_kana=response["name_kana"],
				election_signature=generate_election_signature(response)
			)
			pid = insert_person(cur, new_person)
			print(f"Inserted new person with id: {pid}")

			new_results: list[ElectionResult] = []
			for e in response["election_data"]:
				new_results.append(
					ElectionResult(
						person_id=pid,
						election_date=parse_election_date(e["year"], e["month"], e["day"]),
						election_name=e["election_name"],
						district=e["district"],
						party=e["party"],
						result=e["result"],
						election_number=parse_election_number(e["election_freq"]),
						raw_json=e
					)
				)
			insert_elections_bulk(cur, new_results)
			
			print(f"Deleted {len(id2elections)} duplicates from person table.")
			cur.execute(f"""
			DELETE FROM public.election_result WHERE person_id IN ({",".join([str(id) for id in id2elections.keys()])})
			""")
			print(f"Merged {name} with {len(id2elections)} duplicates.")
			cur.execute(f"""
			DELETE FROM public.person WHERE person_id IN ({",".join([str(id) for id in id2elections.keys()])})
			""")
			print(f"Deleted {len(id2elections)} duplicates from election_result table.")
			break
		except Exception as e:
			raise e

def check_and_merge_duplicates(name:str, ids:List[int], cur:psycopg2.extensions.cursor)->None:

	id2elections = {}
	
	# retrieve the election data for each id
	for id in ids:
		cur.execute(f"""
		SELECT er.* FROM public.election_result er WHERE er.person_id = {id}
		""")
		id2elections[id] = cur.fetchall()

	prompt = compose_duplicate_check_prompt(name, id2elections)
	count = 0
	while True:
		response, _,_ = prompter.prompt(prompt, duplication_check_sys_prompt)
		response = response.strip()
		if response == "SAMEPERSON":
			merge_and_update_db(name, id2elections, cur)
			break
		elif response == "DIFFERENTPERSON":
			break
		count += 1
		if count > 3:
			break

def create_entries_for_politician(name:str, cur:psycopg2.extensions.cursor)->None:
	cur.execute(f"""SELECT * FROM public.person WHERE name_kanji = '{name}'""")
	if cur.fetchone():
		print(f"Person with name {name} already exists in the database.")
		return
	count = 0
	while True:
		try:
			prompt = compose_create_entry_prompt(name, cur)
			response, _,_ = prompter.prompt(prompt, create_entry_sys_prompt)
			response = response.replace("```json", "").replace("```", "")
			response = json.loads(response)
			new_person = Person(
				name_kanji=response["name_kanji"],
				name_kana=response["name_kana"],
				election_signature=generate_election_signature(response)
			)
			pid = insert_person(cur, new_person)
			print(f"Inserted new person with id: {pid}")
			new_results: list[ElectionResult] = []
			for e in response["election_data"]:
				new_results.append(
					ElectionResult(
						person_id=pid,
						election_date=parse_election_date(e["year"], e["month"], e["day"]),
						election_name=e["election_name"],
						district=e["district"],
						party=e["party"],
						result=e["result"],
						election_number=parse_election_number(e["election_freq"]),
						raw_json=e
					)
				)
			insert_elections_bulk(cur, new_results)
			break
		except Exception as e:
			count += 1
			if count > 3:
				raise e
	

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")

    with conn.cursor() as cur:
        create_entries_for = ['百田尚樹']

        for name in create_entries_for:
            create_entries_for_politician(name, cur)
            conn.commit()
		


        duplicates = get_duplicate_politicians(cur)
        print(duplicates)
        groupby_duplicates = create_name2ids_dict(duplicates)
        ignored_names = set([])


        for name, ids in groupby_duplicates.items():
            if name in ignored_names:
                continue
            check_and_merge_duplicates(name, ids, cur)
            conn.commit()
            

            


    print("Done.")

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()

Connected.
Person with name 百田尚樹 already exists in the database.
[(4827, '佐藤静雄', 'さとうしずお', '2ea67e255f8686a2558510022658c70d43c4f1ca31a936f2f6720c4f7cac76ed'), (2624, '佐藤静雄', 'さとうしずお', '89176a6b613c1e69cbe5e64716f4989207140fbd0209ace17bcec295acbb2498'), (5278, '大津力', 'おおつとむ', '7bef6c3e220b9db4ba9642fa929f500aa0c2c3ba6fe42e23e5c10fb548f95217'), (5727, '大津力', 'おおつつとむ', '6be99eedbcee9af6c0c3ff33217b1e5f584e074664e0e9bcb7b323ed52a9f063'), (5724, '寺田学', 'てらたまなぶ', '9a724a4f6fe691e7ab9c35a673e9ce29f1621162a813618d1ea8cc951945a352'), (2603, '寺田学', 'てらたまなぶ、てらだまなぶ', '3006fb7609341e771d442b83a0980e1edd029036e7c4d6cbb9eb1ebdfdbaea8f'), (848, '山田太郎', 'やまだたろう', 'cab913a454539cf5382ee4080e9a2b66713e52cdff716a460afad6ab0d34aef4'), (3819, '山田太郎', 'やまだたろう', '6372817ee1ac790165b7d72b226fb3490baf576faf5960eaaaa4bb4b31149c06'), (5673, '金田勝年', 'かねだかくとし', '17d3fbd11491e09c59089cad6d8171f92fbc18aec9c6b1beacc0d9684624b3f3'), (5728, '金田勝年', 'かねだかつとし', 'c1afe9912783e4d18e13060b41a6ae14e0152bf9efe8b6e1d49dc5a28e8

KeyboardInterrupt: 